In [9]:
import time
import mysql.connector
from collections import defaultdict
from datetime import datetime

# --- Configuration ---
DB_CONFIG = {
    "database": "gossipdb",
    "user": "root",
    "password": "password",
    "host": "localhost",
    "port": 3306  # Default MySQL port
}

In [15]:
import time
import mysql.connector
from datetime import datetime
from collections import defaultdict

# --- CONFIGURATION ---
POLL_INTERVAL = 5 # seconds

def get_db_connection():
    return mysql.connector.connect(**DB_CONFIG)

def format_ts(ts):
    """Safely formats the timestamp into a readable string."""
    # If the timestamp is a huge number, it's likely in milliseconds instead of seconds
    if ts > 1e11:  
        ts = ts
    try:
        return datetime.fromtimestamp(ts).strftime('%H:%M:%S.%f')[:-3]
    except:
        return str(ts)

def get_active_nodes(cursor):
    """Finds all currently active nodes based on their last START action."""
    query = """
        WITH RankedActions AS (
            SELECT 
                CONCAT(host, ':', actionPort) AS node_address, 
                action,
                ROW_NUMBER() OVER(PARTITION BY host, actionPort ORDER BY timestamp DESC) as rn
            FROM ActionRecord
        )
        SELECT node_address 
        FROM RankedActions 
        WHERE rn = 1 AND action = 'START';
    """
    cursor.execute(query)
    return {row[0] for row in cursor.fetchall()}

def get_gossip_timestamps(cursor):
    """
    Extracts the EXACT timestamp each node first saw a specific gossip UID.
    Returns: { uid: { node_address: timestamp, node_address2: timestamp } }
    """
    # This UNION ALL elegantly captures both the creator and forwarder, 
    # fixing the "Creator Ghost" problem at the database level!
    query = """
        SELECT uid, node_address, MIN(timestamp) as min_ts
        FROM (
            SELECT uid, creatorAddress AS node_address, timestamp FROM GossipRecord WHERE creatorAddress IS NOT NULL
            UNION ALL
            SELECT uid, forwarderAddress AS node_address, timestamp FROM GossipRecord WHERE forwarderAddress IS NOT NULL
        ) AS combined
        GROUP BY uid, node_address;
    """
    cursor.execute(query)
    
    gossip_data = defaultdict(dict)
    for row in cursor.fetchall():
        uid, node_address, raw_ts = row
        
        # Handle both MySQL DATETIME objects and raw numbers (Unix epoch)
        if isinstance(raw_ts, datetime):
            ts_val = raw_ts.timestamp() 
        else:
            ts_val = float(raw_ts)
            
        gossip_data[uid][node_address] = ts_val
        
    return gossip_data

def monitor_convergence():
    print("🔍 Starting diagnostic convergence monitor...")
    
    reported_uids = set()
    
    while True:
        try:
            with get_db_connection() as conn:
                with conn.cursor() as cursor:
                    
                    active_nodes = get_active_nodes(cursor)
                    total_active = len(active_nodes)
                    
                    if total_active == 0:
                        print(f"[{time.strftime('%X')}] ⚠️ No active nodes found.")
                        time.sleep(POLL_INTERVAL)
                        continue
                        
                    gossip_data = get_gossip_timestamps(cursor)
                    
                    for uid, node_timestamps in gossip_data.items():
                        active_nodes_with_gossip = {node: ts for node, ts in node_timestamps.items() if node in active_nodes}
                        
                        total_converged = len(active_nodes_with_gossip)
                        
                        if total_converged == total_active:
                            if uid not in reported_uids:
                                first_seen = min(node_timestamps.values())
                                last_seen = max(active_nodes_with_gossip.values())
                                duration = last_seen - first_seen
                                unit = "ms" if first_seen > 1e11 else "seconds"
                                
                                print(f"🎉 [CONVERGED] UID: {uid}")
                                print(f"   ┣ First Seen:         {format_ts(first_seen)}")
                                print(f"   ┣ 100% Converged At:  {format_ts(last_seen)}")
                                print(f"   ┗ Time to Converge:   {duration:.2f} {unit}\n")
                                reported_uids.add(uid)
                        else:
                            # --- DIAGNOSTIC OUTPUT ---
                            if uid not in reported_uids:
                                missing_nodes = active_nodes - set(active_nodes_with_gossip.keys())
                                print(f"⚠️ [STUCK] UID {uid} is at {total_converged}/{total_active} nodes.")
                                print(f"   ┗ Missing on: {', '.join(missing_nodes)}")
                                # Add to reported so we don't spam the console forever
                                reported_uids.add(uid)
                            
        except mysql.connector.Error as err:
            print(f"\n❌ MySQL error: {err}")
        except Exception as e:
            print(f"\n❌ Unexpected error: {e}")
            
        time.sleep(POLL_INTERVAL)

if __name__ == "__main__":
    monitor_convergence()

🔍 Starting diagnostic convergence monitor...
⚠️ [STUCK] UID b3519072-fd2f-4ec8-86f0-135a29331ff9 is at 3/4 nodes.
   ┗ Missing on: localhost:9003
🎉 [CONVERGED] UID: e1aca9c1-b826-fe36-c8b8-eff01365bf10
   ┣ First Seen:         17:38:44.670
   ┣ 100% Converged At:  17:38:44.670
   ┗ Time to Converge:   0.00 seconds

⚠️ [STUCK] UID 6b30dd4d-b692-8898-0e93-23c5078c6d1b is at 3/4 nodes.
   ┗ Missing on: localhost:9002
🎉 [CONVERGED] UID: 6aac3df5-94f5-f077-8310-316fa2184c9c
   ┣ First Seen:         17:38:47.417
   ┣ 100% Converged At:  17:38:47.417
   ┗ Time to Converge:   0.00 seconds

⚠️ [STUCK] UID dac9c252-8ae8-da12-3aa5-5d1a42fa091f is at 3/4 nodes.
   ┗ Missing on: localhost:9003
🎉 [CONVERGED] UID: fc3ca4ba-8c89-c26e-4876-8df142cb63d3
   ┣ First Seen:         17:38:47.672
   ┣ 100% Converged At:  17:38:47.672
   ┗ Time to Converge:   0.00 seconds

🎉 [CONVERGED] UID: ea4aec68-a05a-ad71-f274-aacc66f69692
   ┣ First Seen:         17:38:47.417
   ┣ 100% Converged At:  17:38:47.417
   ┗ Ti

KeyboardInterrupt: 